In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from google.colab import drive, files
from PIL import Image
import matplotlib.pyplot as plt

# Mount your Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_BASE_PATH = "/content/drive/My Drive/NSTV_FDL_Project"
MODEL_SAVE_DIR = os.path.join(PROJECT_BASE_PATH, "Trained_Models")

# Temp path for the uploaded image
UPLOADED_IMAGE_PATH = "/content/uploaded_image.jpg"

print(f"Complete. Using device: {DEVICE}")

Mounting Google Drive...
Mounted at /content/drive
Complete. Using device: cuda


In [2]:
# TransformerNet Class Definition
class ConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super(ConvLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding=kernel_size//2, padding_mode='reflect')
    def forward(self, x): return self.conv(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in1 = nn.InstanceNorm2d(channels, affine=True)
        self.conv2 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in2 = nn.InstanceNorm2d(channels, affine=True)
        self.relu = nn.ReLU()
    def forward(self, x):
        residual = x
        out = self.relu(self.in1(self.conv1(x)))
        out = self.in2(self.conv2(out))
        out = out + residual
        return out

class UpsampleConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, upsample=None):
        super(UpsampleConvLayer, self).__init__()
        self.upsample = upsample
        self.conv = ConvLayer(in_channels, out_channels, kernel_size, stride)
    def forward(self, x):
        if self.upsample:
            x = F.interpolate(x, mode='nearest', scale_factor=self.upsample)
        return self.conv(x)

class TransformerNet(nn.Module):
    def __init__(self):
        super(TransformerNet, self).__init__()
        self.conv1 = ConvLayer(3, 32, kernel_size=9, stride=1)
        self.in1 = nn.InstanceNorm2d(32, affine=True)
        self.conv2 = ConvLayer(32, 64, kernel_size=3, stride=2)
        self.in2 = nn.InstanceNorm2d(64, affine=True)
        self.conv3 = ConvLayer(64, 128, kernel_size=3, stride=2)
        self.in3 = nn.InstanceNorm2d(128, affine=True)
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(128)
        self.res4 = ResidualBlock(128)
        self.res5 = ResidualBlock(128)
        self.deconv1 = UpsampleConvLayer(128, 64, kernel_size=3, stride=1, upsample=2)
        self.in4 = nn.InstanceNorm2d(64, affine=True)
        self.deconv2 = UpsampleConvLayer(64, 32, kernel_size=3, stride=1, upsample=2)
        self.in5 = nn.InstanceNorm2d(32, affine=True)
        self.deconv3 = ConvLayer(32, 3, kernel_size=9, stride=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.relu(self.in1(self.conv1(x)))
        out = self.relu(self.in2(self.conv2(out)))
        out = self.relu(self.in3(self.conv3(out)))
        out = self.res1(out); out = self.res2(out); out = self.res3(out)
        out = self.res4(out); out = self.res5(out)
        out = self.relu(self.in4(self.deconv1(out)))
        out = self.relu(self.in5(self.deconv2(out)))
        out = self.deconv3(out)
        return out

In [3]:
# 1. Load all trained models
print("Loading all trained models...")
model_paths = glob.glob(os.path.join(MODEL_SAVE_DIR, "*.pth"))
models = {}

if not model_paths:
    print(f"Error: No .pth models found in {MODEL_SAVE_DIR}")
    print("Please check the path or train your models first.")
else:
    for path in model_paths:
        try:
            style_name = os.path.basename(path).replace("_net.pth", "")
            model = TransformerNet()

            # Load the state dict, mapping to CPU first
            model.load_state_dict(torch.load(path, map_location=torch.device('cpu')))

            model.to(DEVICE) # Move the model to the correct device
            model.eval()     # Set to evaluation mode
            models[style_name] = model
            print(f"Loaded: {style_name}")
        except Exception as e:
            print(f"Error loading {path}: {e}")

print(f"Complete: {len(models)} models loaded.")

Loading all trained models...
Loaded: War
Loaded: IMAX
Loaded: Cyberpunk_XL
Loaded: Cyberpunk_Mini
Complete: 4 models loaded.


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from PIL import Image
import traceback # Import traceback for detailed errors
import torch # Make sure torch is imported
from torchvision import transforms # Make sure transforms are imported

# Helper Function to run inference
def stylize_image(model, image_path):
    # Open the image from the specified path
    content_image = Image.open(image_path).convert("RGB")

    # Define a safe maximum dimension (width or height)
    max_dim = 1024

    # Calculate new size to maintain aspect ratio
    img_width, img_height = content_image.size
    if img_width > max_dim or img_height > max_dim:
        if img_width > img_height:
            new_width = max_dim
            new_height = int(max_dim * (img_height / img_width))
        else:
            new_height = max_dim
            new_width = int(max_dim * (img_width / img_height))
    else:
        new_width, new_height = img_width, img_height

    # Ensure dimensions are valid for the model
    new_width = new_width - (new_width % 4)
    new_height = new_height - (new_height % 4)

    # Define the transformation with the new resize step
    content_transform = transforms.Compose([
        transforms.Resize((new_height, new_width)), # Resizing fix
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.mul(255))
    ])
    content_tensor = content_transform(content_image).unsqueeze(0).to(DEVICE)

    # Run inference
    with torch.no_grad():
        output_tensor = model(content_tensor)

    # Process the output tensor
    output_data = output_tensor.cpu().squeeze(0).clamp(0, 255).numpy().transpose(1, 2, 0).astype("uint8")
    return Image.fromarray(output_data)

# 1. Check if models are loaded
if not 'models' in globals() or not models:
    print("Models not loaded. Please run Cell 3 first.")
else:
    # 2. Create Widgets
    print("Upload your image, select a style, and click 'Run'.")

    style_options = list(models.keys())
    style_dropdown = widgets.Dropdown(
        options=style_options,
        description='Select Style:',
        value=style_options[0] if style_options else None,
        layout={'width': '500px'}
    )

    uploader = widgets.FileUpload(
        accept='image/*',
        description='Upload Image',
        multiple=False
    )

    run_button = widgets.Button(
        description='Run',
        button_style='success'
    )

    # Output widget to display results
    output_area = widgets.Output()

    # 3. Define the Function to Run on Button Click
    def on_stylize_click(b):
        # Clear previous output and print a status message
        output_area.clear_output()
        with output_area:
            print("Button clicked. Processing...")

        # Get selected style and uploaded file
        selected_style = style_dropdown.value
        uploaded_file_info = uploader.value

        # Input Validation
        if not uploaded_file_info:
            with output_area:
                print("Error: Please upload an image first.")
            return

        if not selected_style:
            with output_area:
                print("Error: No styles loaded. Please check Cell 3.")
            return

        # Process the uploaded file
        try:
            with output_area:
                print("File found. Loading image...")
                filename = list(uploaded_file_info.keys())[0]
                file_content = uploaded_file_info[filename]['content']

                # Save to temp path
                with open(UPLOADED_IMAGE_PATH, 'wb') as f:
                    f.write(file_content)

                print(f"Processing '{filename}' with style '{selected_style}'...")

                model = models[selected_style]

                print("Running inference... (this may take a moment)")
                styled_image = stylize_image(model, UPLOADED_IMAGE_PATH)
                original_image = Image.open(UPLOADED_IMAGE_PATH)
                print("Plotting results...")

                # Display Results Side-by-Side
                plt.figure(figsize=(16, 8))

                ax1 = plt.subplot(1, 2, 1)
                ax1.imshow(original_image)
                ax1.set_title("Original", fontsize=16)
                ax1.axis("off")

                ax2 = plt.subplot(1, 2, 2)
                ax2.imshow(styled_image)
                ax2.set_title(f"Style: {selected_style}", fontsize=16)
                ax2.axis("off")

                plt.show()
                print("Done.")

                # The line that caused the error was here and has been removed

        except Exception as e:
            with output_area:
                print("An error occurred during stylization:")
                print(traceback.format_exc())

    # 4. Link Button to Function
    run_button.on_click(on_stylize_click)

    # 5. Display the UI
    # Group all widgets into one vertical box
    ui_container = widgets.VBox([uploader, style_dropdown, run_button, output_area])

    # Display the single container
    display(ui_container)

Upload your image, select a style, and click 'Run'.
